# Carnot HPR Comparison
This notebook uses the packaged `chocolate_factory.json` case to compare four advanced workflow families on the public `problem.target.*` and `problem.plot.*` surfaces: direct heat pump, indirect heat pump, direct refrigeration, and indirect refrigeration.


## Case context
`chocolate_factory.json` is a realistic multi-zone plant case with enough structure to support HPR screening across multiple load fractions. `PinchWorkspace` keeps the study bundle organized, while each named case still behaves like a real `PinchProblem`.


In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

import pandas as pd

from IPython.display import display

from OpenPinch import PinchWorkspace
from OpenPinch.lib.enums import HPRcycle
from OpenPinch.utils import get_scalar_value

WORKFLOW_LABELS = {
    "direct_heat_pump": "Direct Heat Pump",
    "indirect_heat_pump": "Indirect Heat Pump",
    "direct_refrigeration": "Direct Refrigeration",
    "indirect_refrigeration": "Indirect Refrigeration",
}


In [ ]:
workspace = PinchWorkspace(source="chocolate_factory.json", project_name="Site")
load_fractions = [0.25, 0.5, 0.75, 1.0]

for load_fraction in load_fractions:
    load_case_name = f"load_{int(load_fraction * 100):03d}"
    load_case = workspace.copy_case("baseline", load_case_name, activate=False)
    load_case.update_options(
        {
            "HPR_TYPE": HPRcycle.MultiTempCarnot.value,
            "HPR_LOAD_VALUE": load_fraction,
            "MAX_HP_MULTISTART": 10,
            "N_COND": 3,
            "N_EVAP": 3,
            "REFRIGERANTS": "water;ammonia",
        }
    )
    for workflow_name in WORKFLOW_LABELS:
        workspace.copy_case(
            load_case_name,
            f"{load_case_name}__{workflow_name}",
            activate=False,
        )

{
    "case_count": len(workspace.list_cases()),
    "example_cases": workspace.list_cases()[:8],
}


## Solve one named workflow case through `PinchWorkspace`
The workspace owns the study registry, but the active case is still solved through the normal `PinchProblem` target surface.


In [ ]:
direct_hp_full_case = workspace.case("load_100__direct_heat_pump")
direct_hp_full_case.target.direct_heat_pump()
direct_hp_full_case.summary_frame()


## Inspect the standard HPR-aware plots
Keep the graphing layer on the public plot accessor. The HPR-aware net-load and GCC views appear after the workflow target has been solved.


In [ ]:
profile_problem = workspace.case("load_100__direct_heat_pump")
direct_hp_target = profile_problem.target.direct_heat_pump()

print({"direct_heat_pump_cop": direct_hp_target.hpr_cop})
display(profile_problem.plot.net_load_profiles(zone_name="Direct Heat Pump"))
display(
    profile_problem.plot.grand_composite_curve_with_heat_pump(
        zone_name="Direct Heat Pump"
    )
)


## Compare direct heat-pump cases across load fraction
This keeps one workflow family fixed so you can see how the utility and pinch picture moves as the requested HPR load changes.


In [ ]:
direct_hp_cases = [
    f"load_{int(load_fraction * 100):03d}__direct_heat_pump"
    for load_fraction in load_fractions
]
for case_name in direct_hp_cases:
    workspace.case(case_name).target.direct_heat_pump()

direct_hp_delta_frame = pd.concat(
    [
        workspace.compare_cases(
            direct_hp_cases[0],
            case_name,
            target_name="Site/Direct Heat Pump",
            base_label=direct_hp_cases[0],
            other_label=case_name,
        )
        .loc[["Change"]]
        .assign(variant=case_name)
        for case_name in direct_hp_cases[1:]
    ],
    ignore_index=False,
).reset_index(drop=True)
direct_hp_delta_frame


## Compare all four advanced workflows
Read the workflow labels literally: direct heat pump, indirect heat pump, direct refrigeration, and indirect refrigeration. The comparison table below keeps the same case-generation logic and asks which workflow family gives the best utility and work picture at each load fraction.


In [ ]:
rows = []
for load_fraction in load_fractions:
    for workflow_name, workflow_label in WORKFLOW_LABELS.items():
        case_name = f"load_{int(load_fraction * 100):03d}__{workflow_name}"
        problem = workspace.case(case_name)
        target = getattr(problem.target, workflow_name)()
        rows.append(
            {
                "load fraction": load_fraction,
                "workflow": workflow_label,
                "case": case_name,
                "target": target.name,
                "Qh": get_scalar_value(getattr(target, "Qh", None)),
                "Qc": get_scalar_value(getattr(target, "Qc", None)),
                "Qr": get_scalar_value(getattr(target, "Qr", None)),
                "hpr_work": get_scalar_value(getattr(target, "hpr_work", None)),
                "hpr_cop": get_scalar_value(getattr(target, "hpr_cop", None)),
                "hpr_success": getattr(target, "hpr_success", None),
                "graph_count": len(problem.plot.catalog()),
            }
        )

comparison = pd.DataFrame(rows)
comparison


## Which graph families appear after each workflow?
Use `plot.catalog()` together with the comparison table to see which graph families become available after each workflow target is solved. This stays on the stable graph catalog instead of depending on lower-level cycle internals.


In [ ]:
graph_families = []
for workflow_name, workflow_label in WORKFLOW_LABELS.items():
    case_name = f"load_100__{workflow_name}"
    problem = workspace.case(case_name)
    getattr(problem.target, workflow_name)()
    catalog = problem.plot.catalog()
    graph_families.append(
        {
            "workflow": workflow_label,
            "target": case_name,
            "graph families": ", ".join(sorted(catalog["Graph Type"].unique())),
            "graph count": len(catalog),
        }
    )

pd.DataFrame(graph_families)


## Save and reload the study bundle
This is useful when a multi-case HPR study becomes a reusable engineering artifact rather than a one-off notebook session.


In [ ]:
bundle_workspace = TemporaryDirectory()
bundle_path = Path(bundle_workspace.name) / "chocolate_factory_hpr_workspace.json"
workspace.save_bundle(bundle_path)
reloaded = PinchWorkspace.load_bundle(bundle_path)

{
    "bundle_path": str(bundle_path),
    "case_count": len(reloaded.list_cases()),
    "sample_heat_pump_cases": [
        name for name in reloaded.list_cases() if name.endswith("__direct_heat_pump")
    ][:3],
}


## Interpretation
A good HPR option improves the utility picture at a load that is still operationally plausible. In this workflow, `PinchWorkspace` keeps the named study cases, `compare_cases()` quantifies the deltas, and the standard plot accessors show which graph families become useful after each advanced workflow is solved.
